# Analyses Chapter 5

## setup

In [1]:
import os
import pandas as pd
import pandasai as pai
from pandasai_litellm import LiteLLM
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()  # reads .env from the project root

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://chat-ai.academiccloud.de/v1")
LLM_API_KEY  = os.environ["LLM_API_KEY"]
LLM_MODEL    = os.getenv("LLM_MODEL", "llama-3.3-70b-instruct")

# LiteLLM routes any OpenAI-compatible endpoint via the "openai/" model prefix.
llm = LiteLLM(
    model=f"openai/{LLM_MODEL}",
    api_key=LLM_API_KEY,
    api_base=LLM_BASE_URL,
)

pai.config.set({"llm": llm, "verbose": False, "save_charts": True})

# data
data_path = Path("../data")
df = pd.read_csv( data_path / "slt_raw_data.csv")
col_map = pd.read_csv( data_path / "slt_column_mapping.csv").set_index("original_question")["column"]

# panda ai agent
pai_df = pai.DataFrame(
    df,
    name="slt_survey",
    description=(
        "Socio-Legal Studies survey data (2021). "
        "139 respondents from Germany, the United Kingdom, and other countries. "
        "Covers academic identity, career stage, institutional affiliation, "
        "collaboration patterns, and publication strategies. "
        "Many columns contain NaN for unanswered questions. "
        "Always call .dropna() on a column before grouping, counting, or plotting it."
    ), # type: ignore
)

agent = pai.Agent([pai_df])

## Membership in professional associations

In [2]:
question="""
Column `q15_which_are_the_main_professional` contains free text information on professional associations the respondent belongs to.
Look through each column individually and parse the answers, removing commentary and other noise. Only keep names of associations.
return a table with columns `academic_community` and `associations`.
"""
associations_df = agent.chat(question).value
associations_df.to_csv("../data/associations.csv")

In [ ]:
ass_pai_df = pai.DataFrame(
    associations_df,
    name="slt_survey_associations",
    description=(
        "A table containing unstructured data of professional associations respondents are members of. "
        "The answers are very heterorgenous and must be carefully parsed. "
        "Typically the respondents name several associations, either in full or abbreviated form."
    ), # type: ignore
)

association_list_df =  pai.Agent([ass_pai_df]).chat((
    "Create a list of unique association names mentioned in the data. "
    "Each entry of the list must be one association, so split up values that contain more than one."
)).value
association_list_df

"['RCSL', 'Deutscher Juristinnenbund  Gesellschaft für Rechtsvergleichung  Dt. Sozialrechtsverband  Dt. Familiengerichtstag  Vereinigung für Recht und Gesellschaft  International Society of Family Law  Sozialpolitische Gesellschaft  Bundesarbeitsgemeinschaft Hochschullehrer des Rechts Sozialpädagogik BAG HR e.V.  Gesellschaft für Soziologie Sektion Rechtssoziologie   Deutscher Juristentag e.V.', 'AHRD UFHRD', 'SLS. ALT.   Academy of Social Sciences.  Royal Society of Arts.', 'Deutscher Sozialrechtsverband  Deutscher Sozialgerichtstag  Vereinigung der deutschen Staatsrechtslehrer  Vereinigung für Recht und Gesellschaft  Deutsche Gesellschaft für Rehabilitationswissenschaften', 'Socio-legal Scholars Association  Society of Legal Scholars  Association of Law Teachers', 'Law&Society Association', '•Human Rights Researchers’ Network/Human Rights Consortium  •Doctoral Research Triangle of Human Rights Centres at Essex', 'Crime Studies Network', 'American Society for Legal History', 'BAKJ', '